# Chapter 7: Linked Lists

The programs from Chapter 7 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## Nodes

`linked_list.py`

In [1]:
"""A singly linked list: each node points to the next one."""


class Node:
    __slots__ = ("value", "next")  # saves memory: no per-node dict

    def __init__(self, value, next=None):
        self.value = value
        self.next = next


class LinkedList:
    def __init__(self, items=()):
        self.head = None  # first node
        self.tail = None  # last node, so append() needn't walk the list
        self._n = 0
        for x in items:
            self.append(x)

    def __len__(self):
        return self._n

    def __iter__(self):
        node = self.head
        while node:
            yield node.value
            node = node.next

    def __repr__(self):
        return " -> ".join(map(repr, self)) or "(empty)"

    def push_front(self, value):
        self.head = Node(value, self.head)
        if self.tail is None:
            self.tail = self.head
        self._n += 1

    def append(self, value):
        node = Node(value)
        if self.tail is None:
            self.head = self.tail = node
        else:
            self.tail.next = node
            self.tail = node
        self._n += 1

    def pop_front(self):
        if self.head is None:
            raise IndexError("pop from an empty list")
        node = self.head
        self.head = node.next
        if self.head is None:
            self.tail = None
        self._n -= 1
        return node.value

    def find(self, value):
        """The first node holding value, or None."""
        node = self.head
        while node and node.value != value:
            node = node.next
        return node

    def remove(self, value):
        """Unlink the first node holding value. Raise ValueError if none."""
        before, node = None, self.head
        while node and node.value != value:
            before, node = node, node.next
        if node is None:
            raise ValueError(f"{value!r} not in list")
        if before is None:
            self.head = node.next
        else:
            before.next = node.next
        if node is self.tail:
            self.tail = before
        self._n -= 1

    def reverse(self):
        """Reverse the list in place by turning every arrow around."""
        before, node = None, self.head
        self.tail = node
        while node:
            node.next, before, node = before, node, node.next
        self.head = before

## Reversing in Place

`list_demo.py`

In [2]:
"""The singly linked list in action."""

from linked_list import LinkedList

s = LinkedList(["b", "c"])
s.push_front("a")
s.append("d")
print(s, f"(length {len(s)})")
s.remove("c")
print(s)
s.reverse()
print(s)
print("popped", s.pop_front(), "leaving", s)

'a' -> 'b' -> 'c' -> 'd' (length 4)
'a' -> 'b' -> 'd'
'd' -> 'b' -> 'a'
popped d leaving 'b' -> 'a'


## A Doubly Linked List with a Sentinel

`doubly_linked.py`

In [3]:
"""A doubly linked list with a sentinel node.

The sentinel is a dummy node that sits before the first node and after
the last, so the list is a circle and no operation needs a special case
for an empty list or for the ends.
"""


class DNode:
    __slots__ = ("value", "prev", "next")

    def __init__(self, value=None):
        self.value = value
        self.prev = self.next = self


class DoublyLinkedList:
    def __init__(self, items=()):
        self._sentinel = DNode()
        self._n = 0
        for x in items:
            self.append(x)

    def __len__(self):
        return self._n

    def __iter__(self):
        node = self._sentinel.next
        while node is not self._sentinel:
            yield node.value
            node = node.next

    def _insert_after(self, node, value):
        new = DNode(value)
        new.prev, new.next = node, node.next
        node.next.prev = new
        node.next = new
        self._n += 1
        return new

    def append(self, value):
        return self._insert_after(self._sentinel.prev, value)

    def appendleft(self, value):
        return self._insert_after(self._sentinel, value)

    def unlink(self, node):
        """Remove a node we already hold, in constant time."""
        node.prev.next = node.next
        node.next.prev = node.prev
        self._n -= 1
        return node.value

    def pop(self):
        if not self._n:
            raise IndexError("pop from an empty list")
        return self.unlink(self._sentinel.prev)

    def popleft(self):
        if not self._n:
            raise IndexError("pop from an empty list")
        return self.unlink(self._sentinel.next)

## Three Interview Problems

`list_problems.py`

In [4]:
"""Three classic linked-list interview problems."""

from linked_list import LinkedList, Node


def middle(head):
    """The middle node: a fast pointer moves two steps per slow step."""
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    return slow


def has_cycle(head):
    """Floyd's cycle detection: if the list loops, the fast pointer
    laps the slow one and they meet. Constant extra memory."""
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False


def merge_sorted(a, b):
    """Merge two sorted chains of nodes into one, reusing the nodes."""
    dummy = tail = Node(None)  # a dummy head avoids special cases
    while a and b:
        if a.value <= b.value:
            tail.next, a = a, a.next
        else:
            tail.next, b = b, b.next
        tail = tail.next
    tail.next = a or b
    return dummy.next


def values(head):
    out = []
    while head:
        out.append(head.value)
        head = head.next
    return out


print(middle(LinkedList([1, 2, 3, 4, 5]).head).value)
print(middle(LinkedList([1, 2, 3, 4]).head).value)

looped = LinkedList([1, 2, 3, 4])
print(has_cycle(looped.head))
looped.tail.next = looped.head.next  # 4 now points back to 2
print(has_cycle(looped.head))

merged = merge_sorted(
    LinkedList([1, 4, 6]).head, LinkedList([2, 3, 7]).head
)
print(values(merged))

3
3
False
True
[1, 2, 3, 4, 6, 7]


## When Linked Lists Win, and When They Lose

`bench_front.py`

In [5]:
"""Adding at the front: LinkedList against list and deque."""

from collections import deque

import bench
from linked_list import LinkedList


def make_list(n):
    return (list(range(n)),)


def make_linked(n):
    return (LinkedList(range(n)),)


def make_deque(n):
    return (deque(range(n)),)


def list_front(items):
    items.insert(0, 0)
    items.pop(0)


def linked_front(items):
    items.push_front(0)
    items.pop_front()


def deque_front(items):
    items.appendleft(0)
    items.popleft()


sizes = [10_000, 20_000, 40_000, 80_000, 160_000]
results = {
    "list": bench.doubling(list_front, make_list, sizes),
    "LinkedList": bench.doubling(linked_front, make_linked, sizes),
    "deque": bench.doubling(deque_front, make_deque, sizes),
}
bench.table(results)
bench.chart(results, "ch07_front.png", "Add and remove at the front")

         n            list      LinkedList           deque
    10,000         2.95 us          226 ns         86.9 ns
    20,000    6.62 us x2.2     226 ns x1.0    87.4 ns x1.0
    40,000    13.9 us x2.1     226 ns x1.0    87.9 ns x1.0
    80,000    28.3 us x2.0     228 ns x1.0      86 ns x1.0
   160,000    60.1 us x2.1     225 ns x1.0    90.9 ns x1.1


`bench_walk.py`

In [6]:
"""Walking through every item: LinkedList against list."""

import bench
from linked_list import LinkedList


def walk(items):
    total = 0
    for x in items:
        total += x
    return total


sizes = [10_000, 20_000, 40_000, 80_000]
results = {
    "list": bench.doubling(walk, lambda n: (list(range(n)),), sizes),
    "LinkedList": bench.doubling(
        walk, lambda n: (LinkedList(range(n)),), sizes
    ),
}
bench.table(results)

         n            list      LinkedList
    10,000          143 us          301 us
    20,000     280 us x2.0     619 us x2.1
    40,000     572 us x2.0    1.26 ms x2.0
    80,000    1.52 ms x2.7    2.88 ms x2.3
